# Entrenamiento y comparación de modelos

Requiere `data/processed/tracks_clean.csv` (lo genera `01_exploracion_datos.ipynb`). Todo el código vive en `src/recommender`; este notebook solo lo llama y grafica, así que los números son los mismos que produce `python -m recommender.train`.

1. Elegir K para el modelo B (método del codo y coeficiente de silueta).
2. Comparar modelo A, modelo B y la referencia al azar con las métricas de la Tabla 2.
3. Elegir el modelo final.
   

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.append("../src")

from recommender.config import DEFAULT_K, EVAL_QUERIES, N_CLUSTERS, SEED
from recommender.data import load_clean
from recommender.features import build_preprocessor, to_matrix
from recommender.models import build_model, cluster_scores
from recommender.pipeline import SongRecommender
from recommender.train import compare_models, select_model

CLEAN_PATH = Path("../data/processed/tracks_clean.csv")
BAR_COLOR = "#3B6FB6"

tracks = load_clean(CLEAN_PATH)
features = to_matrix(build_preprocessor().fit(tracks), tracks)
features.shape

## 1. Número de clústeres del modelo B

La inercia siempre baja al aumentar K; buscamos el punto donde deja de bajar rápido (el codo). La silueta va de -1 a 1 y mide qué tan separados quedan los grupos. Se calcula sobre una muestra de 10,000 canciones porque su costo crece con el cuadrado de los datos.

In [ ]:
scores = cluster_scores(features, list(range(2, 21)), seed=SEED)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
ax1.plot(scores["k"], scores["inertia"], marker="o", color=BAR_COLOR)
ax1.set(title="Método del codo", xlabel="K", ylabel="Inercia")
ax2.plot(scores["k"], scores["silhouette"], marker="o", color=BAR_COLOR)
ax2.set(title="Coeficiente de silueta", xlabel="K", ylabel="Silueta")
for ax in (ax1, ax2):
    ax.axvline(N_CLUSTERS, color="gray", linestyle="--", linewidth=1)
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

scores.round(3)

La silueta es baja para cualquier K (alrededor de 0.18): las canciones no forman grupos muy separados, el audio cambia de forma continua entre estilos. Entre K = 4 y K = 8 la silueta casi no cambia, mientras que la inercia sigue bajando hasta K = 8, donde la curva se aplana. Por eso se usa **K = 8** (`N_CLUSTERS` en `config.py`): clústeres más chicos hacen la búsqueda más rápida sin empeorar la separación.

In [ ]:
model_b = build_model("cluster_knn", n_clusters=N_CLUSTERS, seed=SEED).fit(features)
sizes = pd.Series(model_b.labels_).value_counts().sort_index()

cluster_genres = pd.crosstab(model_b.labels_, tracks["macro_genre"], normalize="index")
print("Canciones por clúster:")
print(sizes.to_string())
cluster_genres.round(2)

## 2. Comparación de modelos

Los tres modelos usan el mismo preprocesamiento y la misma muestra de 2,000 canciones de prueba. Para cada una se piden k = 10 recomendaciones.

In [ ]:
results = pd.DataFrame(compare_models(tracks, DEFAULT_K, EVAL_QUERIES, N_CLUSTERS, SEED))
results = results.set_index("model")
results

In [ ]:
precision = f"precision@{DEFAULT_K}"
panels = [
    (precision, "Precision@10 de macro-género (más es mejor)"),
    ("diversity", "Diversidad dentro de la lista"),
    ("popularity_gap", "Sesgo de popularidad (0 = sin sesgo)"),
    ("latency_p95_ms", "Latencia p95 (ms)"),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), constrained_layout=True)
for ax, (column, title) in zip(axes, panels):
    bars = ax.bar(results.index, results[column], color=BAR_COLOR)
    ax.bar_label(bars, fmt="{:.3g}", padding=2)
    ax.set_title(title, loc="left", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 3. Modelo elegido

Regla (`select_model` en `train.py`): gana el de mayor precision@k, pero si dos modelos quedan a menos de un punto (0.01) se consideran empatados y gana el más rápido. La referencia al azar nunca puede ganar.

- Los dos modelos superan claramente a la referencia: aproximadamente 0.32 contra 0.12 de precision@10.
- El modelo B casi no pierde precisión frente al A, porque los vecinos más cercanos de una canción casi siempre caen en su mismo clúster.
- El modelo B es varias veces más rápido (entre 4 y 10 según la máquina y el percentil) porque busca solo dentro de su clúster (~10,000 canciones en promedio) en vez de entre las ~81,000.
- Ninguno muestra sesgo de popularidad: lo recomendado tiene en promedio la misma popularidad que la entrada.
- La diversidad de los modelos es muy baja comparada con el azar. Es lo esperado (recomiendan lo más parecido), pero es el riesgo de "burbuja de filtro" que se discute en el análisis ético.

In [ ]:
selected = select_model(results.reset_index().to_dict("records"), DEFAULT_K)
print("Modelo elegido:", selected)

## 4. Ejemplo

In [ ]:
recommender = SongRecommender.train(tracks, build_model(selected, N_CLUSTERS, SEED))

query = recommender.search("bohemian rhapsody", limit=1)[0]
print(f"Entrada: {query['track_name']} - {query['artists']} ({query['track_genre']})")
pd.DataFrame(recommender.recommend(query["track_id"], k=10))[
    ["track_name", "artists", "track_genre", "popularity", "similarity"]
]

In [ ]:
# Mismas canciones, pero solo con popularidad de 30 o menos (para descubrir artistas nuevos).
pd.DataFrame(recommender.recommend(query["track_id"], k=10, max_popularity=30))[
    ["track_name", "artists", "track_genre", "popularity", "similarity"]
]